In [ ]:
!wget http://aliopentrace.oss-cn-beijing.aliyuncs.com/v2018Traces/batch_instance.tar.gz
!wget http://aliopentrace.oss-cn-beijing.aliyuncs.com/v2018Traces/machine_meta.tar.gz

--2026-03-06 13:44:59--  http://aliopentrace.oss-cn-beijing.aliyuncs.com/v2018Traces/batch_instance.tar.gz
Resolving aliopentrace.oss-cn-beijing.aliyuncs.com (aliopentrace.oss-cn-beijing.aliyuncs.com)... 39.103.20.14
Connecting to aliopentrace.oss-cn-beijing.aliyuncs.com (aliopentrace.oss-cn-beijing.aliyuncs.com)|39.103.20.14|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 21204654955 (20G) [application/gzip]
Saving to: ‘batch_instance.tar.gz’

batch_instance.tar. 100%[===================>]  19.75G  23.0MB/s    in 17m 3s  

2026-03-06 14:02:04 (19.8 MB/s) - ‘batch_instance.tar.gz’ saved [21204654955/21204654955]

--2026-03-06 14:02:05--  http://aliopentrace.oss-cn-beijing.aliyuncs.com/v2018Traces/machine_meta.tar.gz
Resolving aliopentrace.oss-cn-beijing.aliyuncs.com (aliopentrace.oss-cn-beijing.aliyuncs.com)... 39.103.20.14
Connecting to aliopentrace.oss-cn-beijing.aliyuncs.com (aliopentrace.oss-cn-beijing.aliyuncs.com)|39.103.20.14|:80... connected.
HTTP reque

In [ ]:
!tar -xzf batch_instance.tar.gz
!tar -xzf machine_meta.tar.gz

^C


In [ ]:
!tar -xOzf batch_instance.tar.gz batch_instance.csv | head -n 500000 > instance_subset.csv

In [ ]:
!tar -xzf machine_meta.tar.gz


In [ ]:
!ls -lh

total 78G
-rw------- 1 root    root   59G Mar  6 14:14 batch_instance.csv
-rw-r--r-- 1 root    root   20G Feb 16  2023 batch_instance.tar.gz
-rw-r--r-- 1 root    root   40M Mar  6 14:15 instance_subset.csv
-rw-r--r-- 1 1317483 users 540K Dec  3  2018 machine_meta.csv
-rw-r--r-- 1 root    root   91K Feb 15  2023 machine_meta.tar.gz
drwxr-xr-x 1 root    root  4.0K Jan 16 14:24 sample_data


In [ ]:
import pandas as pd

# Define column names based on AliOpenTrace dataset documentation
# For batch_instance.csv (instance_subset.csv)
instance_cols = ['instance_id', 'task_id', 'job_id', 'task_type', 'status', 'start_time', 'end_time', 'machine_id', 'plan_cpu', 'plan_mem', 'cpu_util', 'mem_util', 'disk_io_util', 'network_util']
# For machine_meta.csv
machine_cols = ['machine_id', 'timestamp', 'cpu', 'mem', 'disk', 'status', 'cpu_count', 'mem_size', 'disk_size', 'p1', 'p2', 'p3', 'p4', 'p5', 'p6', 'p7', 'p8', 'p9', 'p10', 'p11', 'p12']

# Load the CSV files, assuming no header in the files and providing column names explicitly
instance = pd.read_csv("instance_subset.csv", header=None, names=instance_cols)
machine = pd.read_csv("machine_meta.csv", header=None, names=machine_cols)

In [ ]:
import pandas as pd
import numpy as np
instance = pd.read_csv("instance_subset.csv")
machine = pd.read_csv("machine_meta.csv")
print(instance.head())
print(machine.head())
print(instance.columns)
print(machine.columns)

    ins_74901673 task_LTg0MTUwNTA5Mjg4MDkwNjIzMA==   j_217  10  Terminated  \
0  ins_815802872                                M1  j_1527   1  Terminated   
1  ins_564677701                                M1  j_2014   1  Terminated   
2  ins_257566161                                M1  j_2014   1  Terminated   
3  ins_688679908                                M1  j_2014   1  Terminated   
4  ins_929638393                                M1  j_2014   1  Terminated   

   673795  673797  m_2637  1  1.1     13     16  0.02  0.02.1  
0  158478  158520  m_3430  1    1    3.0   19.0  0.13    0.18  
1  372602  372616  m_1910  1    1   87.0  116.0  0.04    0.05  
2  372602  372615  m_2485  1    1   91.0  123.0  0.05    0.05  
3  372602  372615   m_993  1    1   93.0  141.0  0.05    0.05  
4  372603  372615  m_2808  1    1  100.0  137.0  0.05    0.05  
   m_1       0  219    17  96  100  USING
0  m_1  148984  219  17.0  96  100  USING
1  m_1  535156  219  17.0  96  100  USING
2  m_1  552384  219  

In [ ]:
# Select and rename columns for instance
instance = instance[['instance_id',
                     'machine_id',
                     'cpu_util', # Using cpu_util from raw data as 'cpu'
                     'mem_util', # Using mem_util from raw data as 'mem'
                     'disk_io_util', # Using disk_io_util from raw data as 'disk'
                     'start_time',
                     'end_time']].rename(columns={
                         'cpu_util': 'cpu',
                         'mem_util': 'mem',
                         'disk_io_util': 'disk'
                     })

# Select columns for machine
machine = machine[['machine_id',
                   'cpu',
                   'mem',
                   'disk']]

instance = instance.dropna()
machine = machine.dropna()

# Convert data types to float
instance['cpu'] = instance['cpu'].astype(float)
instance['mem'] = instance['mem'].astype(float)
instance['disk'] = instance['disk'].astype(float)

machine['cpu'] = machine['cpu'].astype(float)
machine['mem'] = machine['mem'].astype(float)
machine['disk'] = machine['disk'].astype(float)

# Calculate duration
instance['duration'] = instance['end_time'] - instance['start_time']
instance = instance[instance['duration'] > 0]

# Merge dataframes
dataset = pd.merge(instance, machine, on='machine_id', suffixes=('_task','_machine'))
print(dataset.head())

    instance_id machine_id  cpu_task  mem_task  disk_task  start_time  \
0  ins_74901673     m_2637      13.0      16.0       0.02      673795   
1  ins_74901673     m_2637      13.0      16.0       0.02      673795   
2  ins_74901673     m_2637      13.0      16.0       0.02      673795   
3  ins_74901673     m_2637      13.0      16.0       0.02      673795   
4  ins_74901673     m_2637      13.0      16.0       0.02      673795   

   end_time  duration  cpu_machine  mem_machine  disk_machine  
0    673797         2        184.0          6.0          96.0  
1    673797         2        184.0          6.0          96.0  
2    673797         2        184.0          6.0          96.0  
3    673797         2        184.0          6.0          96.0  
4    673797         2        184.0          6.0          96.0  


In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

dataset[['cpu_task','mem_task','disk_task',
         'cpu_machine','mem_machine','disk_machine']] = \
scaler.fit_transform(dataset[['cpu_task','mem_task','disk_task',
                              'cpu_machine','mem_machine','disk_machine']])

In [ ]:
dataset = dataset.sample(n=50000, random_state=42)
print(dataset.shape)

(50000, 11)


In [ ]:
dataset = dataset.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
from sklearn.model_selection import train_test_split
train_data, temp_data = train_test_split(
    dataset,
    test_size=0.30,
    random_state=42
)

In [ ]:
val_data, test_data = train_test_split(
    temp_data,
    test_size=0.50,
    random_state=42
)

In [ ]:
print("Train:", train_data.shape)
print("Validation:", val_data.shape)
print("Test:", test_data.shape)

Train: (35000, 11)
Validation: (7500, 11)
Test: (7500, 11)


In [ ]:
train_data.to_csv("train_dataset.csv", index=False)
val_data.to_csv("validation_dataset.csv", index=False)
test_data.to_csv("test_dataset.csv", index=False)

In [ ]:
from google.colab import files

files.download("train_dataset.csv")
files.download("validation_dataset.csv")
files.download("test_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>